# Buổi 4: Đếm sản phẩm lỗi
**GIẢ LẬP**
## Câu hỏi → dự đoán
Mỗi dòng kết quả đại diện cho gì? Dự đoán số dòng trước khi chạy.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'quality_lab').is_dir() and root!=root.parent: root=root.parent
if not (root/'quality_lab').is_dir(): raise RuntimeError('Mở bằng python run_quality.py --notebook tại thư mục dự án')
sys.path.insert(0,str(root))
from quality_lab.learning import read_sql,learning_db
db=learning_db()
print('Database chỉ đọc:',db)


## SQL mẫu → giải thích
COUNT(*) sau JOIN lỗi đếm dòng loại lỗi. Muốn tính tỷ lệ còn phải có mẫu số tất cả đơn vị đã kiểm tra, gồm đạt.

In [ ]:
read_sql('SELECT i.inspection_no,COUNT(*) AS defect_records,COUNT(DISTINCT i.unit_id) AS defective_units FROM inspections i JOIN inspection_defects d USING(inspection_id) GROUP BY i.inspection_no;')

## Thử thay đổi
So cấp dữ liệu, số dòng và mẫu số của truy vấn thứ hai.

In [ ]:
read_sql('SELECT inspection_no,COUNT(*) AS all_inspected FROM inspections GROUP BY inspection_no;')

## Ví dụ nhỏ: 100 kiểm tra, 8 lỗi, 12 dòng lỗi
Chỉ dùng SQLite trong bộ nhớ, **không ghi DB nguồn**.

In [ ]:
import sqlite3
c=sqlite3.connect(':memory:')
c.executescript('CREATE TABLE inspected(unit_id INTEGER PRIMARY KEY); CREATE TABLE defects(unit_id INTEGER, defect INTEGER);')
c.executemany('INSERT INTO inspected VALUES(?)',[(i,) for i in range(1,101)])
c.executemany('INSERT INTO defects VALUES(?,?)',[(i,1) for i in range(1,9)]+[(i,2) for i in range(1,5)])
lines,units=c.execute('SELECT COUNT(*),COUNT(DISTINCT unit_id) FROM defects').fetchone()
assert (lines,units)==(12,8)
print('12 dòng lỗi, 8 đơn vị lỗi / 100 =',units/100*100,'%')
c.close()

## Tự làm → kiểm tra
Làm bài tương ứng trong docs/exercises.md trước khi đọc solutions.md. Kiểm tra số dòng, mẫu số và NULL.

In [ ]:
assert int(read_sql('SELECT COUNT(*) AS n FROM units').iloc[0,0]) >= int(read_sql('SELECT COUNT(DISTINCT unit_id) AS n FROM inspections WHERE inspection_no=1').iloc[0,0])
print('Số đã kiểm tra đầu không lớn hơn số đơn vị')